1. Set Up

In [1]:
%pip install opencv-python matplotlib numpy 

Note: you may need to restart the kernel to use updated packages.


In [16]:
import cv2
import os
import glob
import shutil
import torch
import numpy as np
from PIL import Image
from tqdm import tqdm
from PIL import Image
from transformers import RTDetrForObjectDetection, AutoImageProcessor

2. Path Configuration

In [25]:
WEIGHTS_PATH = r"D:\Skripsi_Raphaela\rt_detr\program\best_rt_detr_model.pth"
# VIDEO_ROOT = r"D:\Skripsi_Raphaela\conv_lstm\dataset\video_split"
OUTPUT_ROOT = r"D:\Skripsi_Raphaela\conv_lstm\dataset\dataset_split\val\approaching\truck" 
VIDEO_ROOT = r"D:\Skripsi_Raphaela\conv_lstm\dataset\video\New folder"

"""SPLITS = ['train', 'val']
MOTIONS = ['approaching', 'static']
OBJECTS = ['excavator', 'pillar', 'rock', 'traffic cone', 'truck']"""

IMG_SIZE = 112     
FRAME_SKIP = 3     # Extract 1 frame every 3 frames

FOLDER_TO_ID = {
    "excavator": 0,
    "pillar": 1,
    "rock": 2,
    "traffic cone": 3,
    "truck": 4
}
id2label = {v: k for k, v in FOLDER_TO_ID.items()}

# Dynamic confidence thresholds
THRESHOLDS = {0: 0.06, 1: 0.03, 2: 0.04, 3: 0.05, 4: 0.05}

3. Model Initialization

In [18]:
print("Loading RT-DETR architecture and processor from Hugging Face...")

# Load the image processor
processor = AutoImageProcessor.from_pretrained("PekingU/rtdetr_r18vd")

# Load the model architecture configured for 5 custom classes
model = RTDetrForObjectDetection.from_pretrained(
    "PekingU/rtdetr_r18vd",
    num_labels=len(id2label),
    id2label=id2label,
    ignore_mismatched_sizes=True
)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# Load custom trained weights
print(f"Loading custom weights from {WEIGHTS_PATH}...")
# weights_only=True prevents security warnings when loading .pth files
model.load_state_dict(torch.load(WEIGHTS_PATH, map_location=device, weights_only=True))

# Move model to GPU (if available) and set to evaluation mode
model.to(device)
model.eval()

print("Model loaded successfully.")

Loading RT-DETR architecture and processor from Hugging Face...


Loading weights: 100%|██████████| 526/526 [00:00<00:00, 1393.48it/s, Materializing param=model.encoder_input_proj.2.1.weight]                                                 
RTDetrForObjectDetection LOAD REPORT from: PekingU/rtdetr_r18vd
Key                                        | Status   |                                                                                        
-------------------------------------------+----------+----------------------------------------------------------------------------------------
model.decoder.class_embed.{0, 1, 2}.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([80]) vs model:torch.Size([5])          
model.decoder.class_embed.{0, 1, 2}.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([80, 256]) vs model:torch.Size([5, 256])
model.enc_score_head.bias                  | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([80]) vs model:torch.Size([5])          
model.denoising_class_embed.weight       

Using device: cuda
Loading custom weights from D:\Skripsi_Raphaela\rt_detr\program\best_rt_detr_model.pth...
Model loaded successfully.


4. Core Processing Function

In [26]:
"""def process_videos():
    # --- JITTER CONFIGURATION ---
    # If the bounding box center moves less than this many pixels, we lock it in place.
    JITTER_THRESHOLD = 15  

    for split in SPLITS:
        for motion in MOTIONS:
            for obj_name in OBJECTS:
                
                input_dir = os.path.join(VIDEO_ROOT, split, motion, obj_name)
                output_dir = os.path.join(OUTPUT_ROOT, split, motion, obj_name)
                
                if not os.path.exists(input_dir):
                    continue
                    
                video_files = [f for f in os.listdir(input_dir) if f.endswith(('.mp4', '.avi', '.mov'))]
                if not video_files:
                    continue
                    
                print(f"\n[{split.upper()}] Processing: {motion} -> {obj_name} | Found {len(video_files)} videos.")
                
                # Hardware lock the target label based on folder
                target_label_id = FOLDER_TO_ID[obj_name]
                
                for video_name in video_files:
                    video_path = os.path.join(input_dir, video_name)
                    vid_out_dir = os.path.join(output_dir, video_name.split('.')[0])
                    os.makedirs(vid_out_dir, exist_ok=True)
                    
                    cap = cv2.VideoCapture(video_path)
                    frame_idx = 0
                    saved_count = 0
                    last_valid_box = None 
                    
                    while cap.isOpened():
                        ret, frame = cap.read()
                        if not ret: break 
                        
                        if frame_idx % FRAME_SKIP == 0:
                            rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                            pil_img = Image.fromarray(rgb_frame)
                            inputs = processor(images=pil_img, return_tensors="pt").to(device)
                            
                            with torch.no_grad():
                                outputs = model(**inputs)
                            
                            results = processor.post_process_object_detection(
                                outputs, target_sizes=torch.tensor([pil_img.size[::-1]]), threshold=0.01
                            )[0]
                            
                            best_score = 0
                            current_box = None 
                            
                            # 1. WINDOW LOCK: Strictly set based on object type
                            current_fixed_window = 500 if target_label_id == 1 else 400
                            
                            for box, score, label in zip(results["boxes"], results["scores"], results["labels"]):
                                lbl_id = int(label.item())
                                conf = float(score.item())
                                
                                # Match folder label and threshold
                                if lbl_id == target_label_id and conf >= THRESHOLDS.get(lbl_id, 0.5):
                                    if conf > best_score:
                                        best_score = conf 
                                        current_box = box.tolist() 
                                            
                            # 2. POSITION LOCK & ANTI-JITTER LOGIC
                            if current_box is not None:
                                if last_valid_box is None:
                                    last_valid_box = current_box # First detection
                                else:
                                    # Calculate centers
                                    x1, y1, x2, y2 = current_box
                                    lx1, ly1, lx2, ly2 = last_valid_box
                                    
                                    cx, cy = (x1 + x2) / 2, (y1 + y2) / 2
                                    lcx, lcy = (lx1 + lx2) / 2, (ly1 + ly2) / 2
                                    
                                    # If the object barely moved, LOCK THE POSITION (reuse old box)
                                    if abs(cx - lcx) < JITTER_THRESHOLD and abs(cy - lcy) < JITTER_THRESHOLD:
                                        current_box = last_valid_box
                                    else:
                                        last_valid_box = current_box # Update to new position
                                        
                            elif last_valid_box is not None:
                                # Detection failed entirely, fallback to memory
                                current_box = last_valid_box
                            
                            # --- CROP & SAVE ---
                            if current_box is not None:
                                x1, y1, x2, y2 = map(int, current_box)
                                h_frame, w_frame = frame.shape[:2]
                                
                                cx, cy = int((x1 + x2) / 2), int((y1 + y2) / 2)
                                
                                new_x1 = max(0, cx - (current_fixed_window // 2))
                                new_y1 = max(0, cy - (current_fixed_window // 2))
                                new_x2 = min(w_frame, cx + (current_fixed_window // 2))
                                new_y2 = min(h_frame, cy + (current_fixed_window // 2))
                                
                                best_crop = frame[new_y1:new_y2, new_x1:new_x2]
                            
                                if best_crop is not None and best_crop.size > 0:
                                    hh, ww = best_crop.shape[:2]
                                    side = current_fixed_window 
                                    canvas = np.zeros((side, side, 3), dtype=np.uint8)
                                    
                                    y_off = (side - hh) // 2
                                    x_off = (side - ww) // 2
                                    
                                    canvas[y_off:y_off+hh, x_off:x_off+ww] = best_crop
                                    final_img = cv2.resize(canvas, (IMG_SIZE, IMG_SIZE))
                                    
                                    save_path = os.path.join(vid_out_dir, f"frame_{saved_count:04d}.jpg")
                                    cv2.imwrite(save_path, final_img)
                                    saved_count += 1
                                
                        frame_idx += 1
                    cap.release()
                    print(f"   -> {video_name} : {saved_count} frames saved.")
                    """ 

def process_additional_videos():
    # --- JITTER CONFIGURATION ---
    JITTER_THRESHOLD = 15  
    
    # Hard-lock the label to 'truck' since we know exactly what these videos are
    target_label_id = FOLDER_TO_ID["truck"] 
    
    if not os.path.exists(VIDEO_ROOT):
        print(f"Error: Could not find the input folder {VIDEO_ROOT}")
        return
        
    os.makedirs(OUTPUT_ROOT, exist_ok=True)
        
    video_files = [f for f in os.listdir(VIDEO_ROOT) if f.endswith(('.mp4', '.avi', '.mov'))]
    if not video_files:
        print("No videos found in the additional folder.")
        return
        
    print(f"\nProcessing {len(video_files)} additional videos directly to Val -> Approaching -> Truck...")
    
    for video_name in video_files:
        video_path = os.path.join(VIDEO_ROOT, video_name)
        vid_out_dir = os.path.join(OUTPUT_ROOT, video_name.split('.')[0])
        os.makedirs(vid_out_dir, exist_ok=True)
        
        cap = cv2.VideoCapture(video_path)
        frame_idx = 0
        saved_count = 0
        last_valid_box = None 
        
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret: break 
            
            if frame_idx % FRAME_SKIP == 0:
                rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                pil_img = Image.fromarray(rgb_frame)
                inputs = processor(images=pil_img, return_tensors="pt").to(device)
                
                with torch.no_grad():
                    outputs = model(**inputs)
                
                results = processor.post_process_object_detection(
                    outputs, target_sizes=torch.tensor([pil_img.size[::-1]]), threshold=0.01
                )[0]
                
                best_score = 0
                current_box = None 
                
                # Window size is 400 for trucks
                current_fixed_window = 400
                
                for box, score, label in zip(results["boxes"], results["scores"], results["labels"]):
                    lbl_id = int(label.item())
                    conf = float(score.item())
                    
                    # Match folder label and threshold
                    if lbl_id == target_label_id and conf >= THRESHOLDS.get(lbl_id, 0.5):
                        if conf > best_score:
                            best_score = conf 
                            current_box = box.tolist() 
                                
                # POSITION LOCK & ANTI-JITTER LOGIC
                if current_box is not None:
                    if last_valid_box is None:
                        last_valid_box = current_box 
                    else:
                        x1, y1, x2, y2 = current_box
                        lx1, ly1, lx2, ly2 = last_valid_box
                        
                        cx, cy = (x1 + x2) / 2, (y1 + y2) / 2
                        lcx, lcy = (lx1 + lx2) / 2, (ly1 + ly2) / 2
                        
                        if abs(cx - lcx) < JITTER_THRESHOLD and abs(cy - lcy) < JITTER_THRESHOLD:
                            current_box = last_valid_box
                        else:
                            last_valid_box = current_box 
                            
                elif last_valid_box is not None:
                    current_box = last_valid_box
                
                # --- CROP & SAVE ---
                if current_box is not None:
                    x1, y1, x2, y2 = map(int, current_box)
                    h_frame, w_frame = frame.shape[:2]
                    
                    cx, cy = int((x1 + x2) / 2), int((y1 + y2) / 2)
                    
                    new_x1 = max(0, cx - (current_fixed_window // 2))
                    new_y1 = max(0, cy - (current_fixed_window // 2))
                    new_x2 = min(w_frame, cx + (current_fixed_window // 2))
                    new_y2 = min(h_frame, cy + (current_fixed_window // 2))
                    
                    best_crop = frame[new_y1:new_y2, new_x1:new_x2]
                
                    if best_crop is not None and best_crop.size > 0:
                        hh, ww = best_crop.shape[:2]
                        side = current_fixed_window 
                        canvas = np.zeros((side, side, 3), dtype=np.uint8)
                        
                        y_off = (side - hh) // 2
                        x_off = (side - ww) // 2
                        
                        canvas[y_off:y_off+hh, x_off:x_off+ww] = best_crop
                        final_img = cv2.resize(canvas, (IMG_SIZE, IMG_SIZE))
                        
                        save_path = os.path.join(vid_out_dir, f"frame_{saved_count:04d}.jpg")
                        cv2.imwrite(save_path, final_img)
                        saved_count += 1
                    
            frame_idx += 1
        cap.release()
        print(f"   -> {video_name} : {saved_count} frames saved.")



5. Execution

In [28]:
""" process_videos()
print("\n[DONE] Image dataset extraction is complete!")""" 

# Run the function
process_additional_videos()


Processing 5 additional videos directly to Val -> Approaching -> Truck...
   -> VID_20260226_124254.mp4 : 33 frames saved.
   -> VID_20260226_124306.mp4 : 43 frames saved.
   -> VID_20260226_124319.mp4 : 56 frames saved.
   -> VID_20260226_124336.mp4 : 38 frames saved.
   -> VID_20260226_124349.mp4 : 43 frames saved.
